<a href="https://colab.research.google.com/github/Kaunaingul-ai/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kaunaingul-ai/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

##Method choice: Random Forest classifier

I will use a Random Forest classifier to estimate the probability that a content page is declining. This fits my lane because the earlier signal checks showed that relationships between staleness, CTR, search position, and decline are not simple or strictly linear. A Random Forest can capture these interactions without requiring a complex feature transformation.

I will keep the model relatively small and compare it directly with my Week-4 rule-based baseline. The goal is not to reward complexity, but to test whether a learned model produces a better ranking of pages for review.

In [5]:
import os
import subprocess
import pandas as pd
import numpy as np

REPO_URL = "https://github.com/Kaunaingul-ai/flyrank-ml-internship"
REPO_DIR = "/content/flyrank-ml-internship"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

features = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

target = (df["trend_direction"] == "down").astype(int)

print("Rows:", len(df))
print("Features:", features)
print("Declining rate:", round(target.mean(), 3))

Rows: 30000
Features: ['impressions_90d', 'ctr', 'avg_position', 'content_age_days', 'days_since_last_update', 'word_count']
Declining rate: 0.542


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

##Split design

I will use a stratified train/validation split, with 80% of the rows used for training and 20% held out for validation. Stratification keeps the proportion of declining and non-declining pages similar in both sets.

This split is appropriate for this starter dataset because each row represents an anonymized content page and there is no client identifier being used in the model. The held-out validation set will be used only for evaluation. I will compare both the Week-4 baseline and the Random Forest on this same validation set and using the same ranking metric.

In [6]:
from sklearn.model_selection import train_test_split

X = df[features].copy()
y = target.copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))
print("Training decline rate:", round(y_train.mean(), 3))
print("Validation decline rate:", round(y_valid.mean(), 3))


Training rows: 24000
Validation rows: 6000
Training decline rate: 0.542
Validation decline rate: 0.542


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

##Model and baseline comparison

I will compare the Random Forest with my Week-4 baseline on the same 6,000 held-out validation rows. Both methods will be evaluated using Precision@50, which measures the proportion of truly declining pages among the 50 pages ranked highest for review.

The Week-4 baseline uses the same rule as before: one point for staleness and one point for low CTR relative to position. The Random Forest produces a probability of decline. Comparing both rankings on the same validation set makes the comparison consistent.

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score

# ---------- Common metric ----------
def precision_at_k(scores, y_true, k=50):
    scores = np.asarray(scores)
    y_true = np.asarray(y_true)

    top_idx = np.argsort(scores)[::-1][:k]
    return y_true[top_idx].mean()


# ---------- Week-4 baseline on validation set ----------
baseline_valid = X_valid.copy()

baseline_valid["is_stale"] = (
    baseline_valid["days_since_last_update"] >= 90
).astype(int)

baseline_valid["low_ctr_for_position"] = (
    (baseline_valid["avg_position"] >= 4)
    & (baseline_valid["avg_position"] <= 20)
    & (baseline_valid["ctr"] < 0.10)
).astype(int)

baseline_valid["baseline_score"] = (
    baseline_valid["is_stale"]
    + baseline_valid["low_ctr_for_position"]
)

# Tiny tie-break using impressions, as in Week 4 ranking
baseline_rank_score = (
    baseline_valid["baseline_score"]
    + baseline_valid["impressions_90d"].rank(pct=True) * 0.001
)


# ---------- Random Forest ----------
rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_score = rf_model.predict_proba(X_valid)[:, 1]


# ---------- Same metric, same validation rows ----------
baseline_p50 = precision_at_k(
    baseline_rank_score,
    y_valid,
    k=50
)

rf_p50 = precision_at_k(
    rf_score,
    y_valid,
    k=50
)

comparison = pd.DataFrame({
    "Method": ["Week-4 baseline", "Random Forest"],
    "Precision@50": [baseline_p50, rf_p50]
})

print(comparison.round(3))

print("\nRandom Forest secondary metrics:")
print("Average Precision:", round(average_precision_score(y_valid, rf_score), 3))
print("ROC AUC:", round(roc_auc_score(y_valid, rf_score), 3))


            Method  Precision@50
0  Week-4 baseline          0.60
1    Random Forest          0.88

Random Forest secondary metrics:
Average Precision: 0.733
ROC AUC: 0.723


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

##Errors and interpretation

The Random Forest improved Precision@50 from 0.60 for the Week-4 baseline to 0.88 on the same validation set. This means that 44 of the model's top 50 recommendations were actually declining, compared with 30 of the baseline's top 50.

However, the model is not perfect. Some highly ranked pages are false positives, meaning the model assigns them a high decline probability even though their observed outcome is not decline. It can also miss declining pages by assigning them lower scores. These errors may occur because the available features do not capture every reason why a page changes performance.

I will inspect the model's most important features and some of its false-positive recommendations to understand what signals it relies on. Feature importance should be interpreted as model reliance rather than causal evidence.

In [8]:
# ---------- Feature importance ----------
rf_estimator = rf_model.named_steps["model"]

importance_table = pd.DataFrame({
    "feature": features,
    "importance": rf_estimator.feature_importances_
}).sort_values("importance", ascending=False)

print("Random Forest feature importance:")
display(importance_table)


# ---------- Inspect top-50 model errors ----------
validation_review = X_valid.copy()
validation_review["actual_decline"] = y_valid.values
validation_review["rf_score"] = rf_score

validation_review = validation_review.sort_values(
    "rf_score",
    ascending=False
)

top50 = validation_review.head(50).copy()

false_positives = top50[
    top50["actual_decline"] == 0
]

print("\nTop-50 recommendations:", len(top50))
print("True declining pages in top 50:", int(top50["actual_decline"].sum()))
print("False positives in top 50:", len(false_positives))

print("\nExample false positives:")
display(
    false_positives[
        [
            "impressions_90d",
            "ctr",
            "avg_position",
            "content_age_days",
            "days_since_last_update",
            "word_count",
            "rf_score",
            "actual_decline"
        ]
    ].head(10)
)

Random Forest feature importance:


,feature,importance
0,impressions_90d,0.361094
2,avg_position,0.243414
3,content_age_days,0.215163
5,word_count,0.076285
1,ctr,0.053920
4,days_since_last_update,0.050125



Top-50 recommendations: 50
True declining pages in top 50: 44
False positives in top 50: 6

Example false positives:


,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update,word_count,rf_score,actual_decline
7882,908,0.11,27.4,144,104,1381.0,0.764794,0
24397,187,0.00,28.0,165,104,1306.0,0.763504,0
2164,945,0.11,5.5,139,104,1442.0,0.758306,0
22840,247,0.00,16.8,165,104,1602.0,0.752209,0
16643,175,0.00,26.5,165,104,1672.0,0.747875,0
12595,611,0.00,38.1,155,104,1460.0,0.743687,0


##Interpretation of the model

The Random Forest produced 44 true declining pages among its top 50 recommendations, leaving 6 false positives. This is a clear improvement over the Week-4 baseline, which achieved Precision@50 of 0.60 on the same validation rows.

The model relied most strongly on impressions_90d, avg_position, and content_age_days. This suggests that visibility, search position, and content age were useful signals for ranking pages in this dataset. However, feature importance only shows which variables the model relied on; it does not prove that these features cause decline.

The false-positive examples also show why human review is still necessary. Some pages received relatively high decline scores even though their observed outcome was not decline. This may happen because the model does not have all relevant context, such as seasonality, page purpose, query intent, or business priorities. I therefore treat the model as a decision-support tool rather than an automatic decision-maker.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.